In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch

In [2]:
import torch
import numpy as np
from typing import Any, Type
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_curve, auc
from torch.utils.data import Dataset, DataLoader
import tqdm

Tensor = Type[torch.Tensor]


class EarlyStopping:
    """早停模块，用于防止过拟合"""
    def __init__(self, patience=7, delta=0, verbose=False, path='checkpoint.pt'):
        """
        Args:
            patience (int): 在多少个epoch没有改善后停止训练
            delta (float): 改善的最小变化量
            verbose (bool): 是否打印信息
            path (str): 保存最佳模型的路径
        """
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.path = path
        
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.best_auc = 0
        
    def __call__(self, val_loss, val_auc, model):
        score = -val_loss  # 我们想要最小化损失
        
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, val_auc, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, val_auc, model)
            self.counter = 0
            
    def save_checkpoint(self, val_loss, val_auc, model):
        """保存最佳模型"""
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        
        # 保存整个模型
        torch.save({
            'model_state_dict': model.state_dict(),
            'val_loss': val_loss,
            'val_auc': val_auc
        }, self.path)
        
        self.val_loss_min = val_loss
        self.best_auc = val_auc


class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
        self.index = -1

    def forward(self, x: Tensor) -> Tensor:
        """
        Arguments:
            x: Tensor, shape ``[seq_len, batch_size, embedding_dim]``
        """
        self.index = -1
        self.index += 1
        return x + self.pe[self.index:x.size(0)+self.index]


class Transformer(torch.nn.Module):
    def __init__(self,
        src_modalities: dict[str, dict[str, Any]],
        tgt_modalities: dict[str, dict[str, Any]],
        d_model: int,
        nhead: int,
        num_encoder_layers: int = 1,
        device: str = 'cpu',
        dropout: float = 0.3,  # 添加dropout参数
        use_layer_norm: bool = True  # 添加LayerNorm选项
    ) -> None:
        super().__init__()

        self.d_model = d_model
        self.nhead = nhead
        self.num_encoder_layers = num_encoder_layers
        self.src_modalities = src_modalities
        self.tgt_modalities = tgt_modalities
        self.device = device

        # 特征嵌入层
        self.modules_emb_src = torch.nn.ModuleDict()
        for k, info in src_modalities.items():
            if info['type'] == 'numerical':
                layers = [
                    torch.nn.BatchNorm1d(info['shape']),
                    torch.nn.Dropout(dropout/2),  # 嵌入层dropout
                    torch.nn.Linear(info['shape'], d_model)
                ]
                if use_layer_norm:
                    layers.append(torch.nn.LayerNorm(d_model))
                self.modules_emb_src[k] = torch.nn.Sequential(*layers)
            else:
                raise ValueError(f'{k} is an unrecognized data modality')
            
        # 位置编码
        self.pe = PositionalEncoding(d_model)

        # 目标嵌入向量
        self.emb_aux = torch.nn.Parameter(
            torch.zeros(len(tgt_modalities), 1, d_model),
            requires_grad = True,
        )
        
        # Transformer编码器
        enc = torch.nn.TransformerEncoderLayer(
            self.d_model, self.nhead,
            dim_feedforward = self.d_model * 4,  # 增加前馈网络维度
            activation = 'gelu',
            dropout = dropout,
            batch_first=False,  # Transformer默认输入是(seq_len, batch_size, embedding_dim)
            norm_first=True  # 使用Pre-LN结构，更稳定
        )
        self.transformer = torch.nn.TransformerEncoder(
            enc, 
            self.num_encoder_layers,
            norm=torch.nn.LayerNorm(d_model) if use_layer_norm else None
        )

        # 分类器 - 添加dropout和更多层
        self.modules_cls = torch.nn.ModuleDict()
        for k, info in tgt_modalities.items():
            if info['type'] == 'binary':
                # 使用更复杂的分类器，但保持正则化
                self.modules_cls[k] = torch.nn.Sequential(
                    torch.nn.Dropout(dropout),
                    torch.nn.Linear(d_model, d_model // 2),
                    torch.nn.LayerNorm(d_model // 2) if use_layer_norm else torch.nn.Identity(),
                    torch.nn.GELU(),
                    torch.nn.Dropout(dropout/2),
                    torch.nn.Linear(d_model // 2, 1)
                )
            else:
                raise ValueError(f'Unsupported target modality type: {info["type"]}')
    def forward(self,
        x: dict[str, Tensor],
        mask: dict[str, Tensor],
    ) -> dict[str, Tensor]:
        """前向传播"""
        # 嵌入层
        out_emb = self.forward_emb(x, mask)
        # Transformer层
        out_trf = self.forward_trf(out_emb, mask)
        # 分类层
        out_cls = self.forward_cls(out_trf)
        
        return out_cls

    def forward_emb(self, x: dict[str, Tensor], mask: dict[str, Tensor]) -> dict[str, Tensor]:
        """嵌入层前向传播"""
        out_emb = dict()
        for k in self.modules_emb_src.keys():
            if not torch.all(mask[k]):
                # 确保输入是2D的
                if self.src_modalities[k]['type'] == 'numerical':
                    if x[k].dim() == 1:
                        x_input = x[k].unsqueeze(1)  # 将1D张量转换为2D
                    else:
                        x_input = x[k]
                    out_emb[k] = self.modules_emb_src[k](x_input)
                else:
                    out_emb[k] = self.modules_emb_src[k](x[k])
            else:
                if 'cuda' in self.device:
                    device = x[k].device
                else:
                    device = self.device
                out_emb[k] = torch.zeros((mask[k].shape[0], self.d_model)).to(device, non_blocking=True)
        return out_emb

    def forward_trf(self,
        out_emb: dict[str, Tensor],
        mask: dict[str, Tensor],
    ) -> dict[str, Tensor]:
        """Transformer层前向传播"""
        N = len(next(iter(out_emb.values())))  # batch size
        S = len(self.modules_emb_src)  # number of sources
        T = len(self.modules_cls)  # number of targets
        src_iter = self.modules_emb_src.keys()
        
        # 堆叠所有源嵌入
        emb_src = torch.stack([o for o in out_emb.values()], dim=0)
        
        # 添加位置编码
        emb_src = self.pe(emb_src)
        
        # 目标嵌入
        emb_tgt = self.emb_aux.repeat(1, N, 1)
        
        # 连接源嵌入和目标嵌入
        emb_all = torch.cat((emb_tgt, emb_src), dim=0)

        # 组合掩码
        mask_src = [mask[k] for k in src_iter]
        mask_src = torch.stack(mask_src, dim=1)
        
        # 目标掩码
        mask_tgt = torch.zeros((N, T), dtype=torch.bool, device=self.emb_aux.device)
        
        # 连接源掩码和目标掩码
        mask_all = torch.cat((mask_tgt, mask_src), dim=1)
        
        # 重复掩码以适应transformer
        mask_all = mask_all.unsqueeze(1).expand(-1, S + T, -1).repeat(self.nhead, 1, 1)

        # 运行transformer
        out_trf = self.transformer(
            src = emb_all,
            mask = mask_all,
        )[0]
        return out_trf

    def forward_cls(self,
        out_trf: dict[str, Tensor],
    ) -> dict[str, Tensor]:
        """分类层前向传播"""
        tgt_iter = self.modules_cls.keys()
        out_cls = {k: self.modules_cls[k](out_trf).squeeze(1) for k in tgt_iter}
        return out_cls

class SigmoidFocalLoss(torch.nn.Module):
    """Sigmoid focal loss for binary classification"""
    def __init__(self, alpha=-1, gamma=2.0, reduction='none'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        """
        Arguments:
            inputs: Tensor of arbitrary shape
            targets: Tensor of the same shape as inputs
        """
        probs = torch.sigmoid(inputs)
        ce_loss = torch.nn.functional.binary_cross_entropy_with_logits(
            inputs, targets, reduction="none"
        )
        
        p_t = probs * targets + (1 - probs) * (1 - targets)
        loss = ce_loss * ((1 - p_t) ** self.gamma)
        
        if self.alpha >= 0:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            loss = alpha_t * loss
            
        if self.reduction == "mean":
            loss = loss.mean()
        elif self.reduction == "sum":
            loss = loss.sum()
            
        return loss

class IrisDataset(Dataset):
    """数据集类，支持数值和分类特征"""
    def __init__(self, features, labels, src_modalities, dropout_rate=0.1):
        """
        初始化数据集
        
        Args:
            features: 列表形式的特征，每个元素是一个字典，包含所有特征
            labels: 列表形式的标签，每个元素是一个字典，包含标签
            src_modalities: 特征模态字典，描述每种特征的类型
            dropout_rate: 特征随机遮蔽率
        """
        self.features = features
        self.labels = labels
        self.src_modalities = src_modalities
        self.dropout_rate = dropout_rate
        
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        # 准备特征 - 加入随机遮蔽
        x = {}
        mask = {}
        
        # 处理每个特征
        for feature_name, feature_value in self.features[idx].items():
            # 根据特征类型处理
            if self.src_modalities[feature_name]['type'] == 'numerical':
                # 数值特征
                if np.isnan(feature_value):
                    x[feature_name] = torch.tensor([0.0], dtype=torch.float32)
                    mask[feature_name] = torch.tensor(True)  # 标记为掩码
                else:
                    x[feature_name] = torch.tensor([feature_value], dtype=torch.float32)
                    # 随机决定是否遮蔽该特征
                    if np.random.random() < self.dropout_rate:
                        mask[feature_name] = torch.tensor(True)
                    else:
                        mask[feature_name] = torch.tensor(False)
            # elif self.src_modalities[feature_name]['type'] == 'categorical':
            #     # 分类特征
            #     if np.isnan(feature_value):
            #         x[feature_name] = torch.tensor(0, dtype=torch.long)  # 使用0作为缺失值
            #         mask[feature_name] = torch.tensor(True)  # 标记为掩码
            #     else:
            #         x[feature_name] = torch.tensor(int(feature_value), dtype=torch.long)
            #         # 随机决定是否遮蔽该特征
            #         if np.random.random() < self.dropout_rate:
            #             mask[feature_name] = torch.tensor(True)
            #         else:
            #             mask[feature_name] = torch.tensor(False)
        
        # 准备标签
        y = {}
        y_mask = {}
        label_key = next(iter(self.labels[idx].keys()))  # 获取第一个标签的键名（通常是'is_depressed'）
        for k, v in self.labels[idx].items():
            y[k] = torch.tensor(v, dtype=torch.float32)
            y_mask[k] = torch.tensor(1.0)  # 不遮蔽标签
        
        return x, y, mask, y_mask
    @staticmethod
    def collate_fn(batch):
        x_batch = {}
        y_batch = {}
        mask_batch = {}
        y_mask_batch = {}
        
        for x, y, mask, y_mask in batch:
            for k in x:
                if k not in x_batch:
                    x_batch[k] = []
                x_batch[k].append(x[k])
                
            for k in y:
                if k not in y_batch:
                    y_batch[k] = []
                y_batch[k].append(y[k])
                
            for k in mask:
                if k not in mask_batch:
                    mask_batch[k] = []
                mask_batch[k].append(mask[k])
                
            for k in y_mask:
                if k not in y_mask_batch:
                    y_mask_batch[k] = []
                y_mask_batch[k].append(y_mask[k])
        
        # 转换为tensor
        for k in x_batch:
            x_batch[k] = torch.cat(x_batch[k], dim=0)
        
        for k in y_batch:
            y_batch[k] = torch.tensor(y_batch[k])
            
        for k in mask_batch:
            mask_batch[k] = torch.stack(mask_batch[k])
            
        for k in y_mask_batch:
            y_mask_batch[k] = torch.tensor(y_mask_batch[k])
            
        return x_batch, y_batch, mask_batch, y_mask_batch

class SimpleADRDModel():
    """简化版ADRDModel"""
    def __init__(self, 
                src_modalities, 
                tgt_modalities, 
                label_fractions,
                d_model=32,
                nhead=1,
                num_encoder_layers=2,
                num_epochs=50,
                batch_size=16,
                lr=1e-3,
                weight_decay=0.01,
                criterion='AUC (ROC)',
                device='cpu',
                verbose=1,
                dropout_rate=0.1,
                ranking_loss=True,
                # 添加早停参数
                early_stopping_patience=10,
                early_stopping_delta=0.001,
                # 添加正则化参数
                dropout_transformer=0.3,
                label_smoothing=0.0,
                # 添加训练策略参数
                warmup_epochs=5,
                lr_min=1e-6): 
        
        self.src_modalities = src_modalities
        self.tgt_modalities = tgt_modalities
        self.label_fractions = label_fractions
        self.d_model = d_model
        self.nhead = nhead
        self.num_encoder_layers = num_encoder_layers
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.criterion = criterion
        self.device = device
        self.verbose = verbose
        self.dropout_rate = dropout_rate
        self.ranking_loss = ranking_loss
        self.dropout_transformer = dropout_transformer
        self.label_smoothing = label_smoothing
        self.warmup_epochs = warmup_epochs
        self.lr_min = lr_min
        
        # 初始化ranking loss参数
        self.lambda_coeff = 0.005
        self.margin = 0.25
        self.margin_loss = torch.nn.MarginRankingLoss(reduction='sum', margin=self.margin)
        
        # 初始化网络
        self.net_ = Transformer(
            src_modalities=self.src_modalities,
            tgt_modalities=self.tgt_modalities,
            d_model=self.d_model,
            nhead=self.nhead,
            num_encoder_layers=self.num_encoder_layers,
            device=self.device
        )
        self.net_.to(self.device)
        
        # 初始化优化器
        self.optimizer = torch.optim.AdamW(
            self.net_.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay
        )
        
        # 初始化学习率调度器 - 使用余弦退火
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=num_epochs - warmup_epochs,
            eta_min=lr_min
        )
        
        # 初始化早停
        self.early_stopping = EarlyStopping(
            patience=early_stopping_patience,
            delta=early_stopping_delta,
            verbose=verbose > 0,
            path='best_model.pt'
        )
        
        # 初始化损失函数
        self.loss_fn = {}
        for k in self.tgt_modalities:
            if self.label_fractions[k] >= 0.3:
                alpha = -1
            else:
                alpha = pow((1 - self.label_fractions[k]), 2)
                
            self.loss_fn[k] = SigmoidFocalLoss(
                alpha=alpha,
                gamma=2.0,
                reduction='none'
            )
        
        # 存储训练历史
        self.train_history = {
            'train_loss': [],
            'train_auc': [],
            'val_loss': [],
            'val_auc': []
        }
    
    def fit(self, x_trn, x_vld, y_trn, y_vld, save_path='model_checkpoint'):
        """训练模型"""
        # 创建数据加载器
        train_dataset = IrisDataset(x_trn, y_trn, self.src_modalities)
        val_dataset = IrisDataset(x_vld, y_vld, self.src_modalities)
        
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            collate_fn=IrisDataset.collate_fn
        )
        
        val_loader = DataLoader(
            val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=IrisDataset.collate_fn
        )
        
        # 记录最佳验证性能
        best_val_metric = 0
        best_epoch = 0
        
        # 训练循环
        for epoch in range(self.num_epochs):
            # 热身阶段调整学习率
            if epoch < self.warmup_epochs:
                lr_scale = min(1., float(epoch + 1) / self.warmup_epochs)
                for param_group in self.optimizer.param_groups:
                    param_group['lr'] = self.lr * lr_scale
            
            # 训练
            train_metrics = self.train_one_epoch(train_loader, epoch)
            # 验证
            val_metrics = self.validate_one_epoch(val_loader)
            
            # 更新学习率（跳过热身阶段）
            if epoch >= self.warmup_epochs:
                self.scheduler.step()
            
            # 记录历史
            self.train_history['train_loss'].append(train_metrics['Loss'])
            self.train_history['train_auc'].append(train_metrics['AUC'])
            self.train_history['val_loss'].append(val_metrics['Loss'])
            self.train_history['val_auc'].append(val_metrics['AUC'])
            
            # 打印性能
            if self.verbose >= 1:
                current_lr = self.optimizer.param_groups[0]['lr']
                print(f"Epoch {epoch+1}/{self.num_epochs}, LR: {current_lr:.6f}")
                print(f"Train - Loss: {train_metrics['Loss']:.4f}, AUC: {train_metrics['AUC']:.4f}")
                print(f"Val   - Loss: {val_metrics['Loss']:.4f}, AUC: {val_metrics['AUC']:.4f}")
            
            # 使用早停
            self.early_stopping(val_metrics['Loss'], val_metrics['AUC'], self.net_)
            
            # 保存最佳模型
            if val_metrics[self.criterion] > best_val_metric:
                best_val_metric = val_metrics[self.criterion]
                best_epoch = epoch
                self.best_model_state = {k: v.cpu().clone() for k, v in self.net_.state_dict().items()}
                # 保存到文件
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.net_.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_auc': val_metrics['AUC'],
                    'train_history': self.train_history
                }, f'{save_path}_best.pth')
            
            if self.early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break
        
        print(f"Best validation {self.criterion}: {best_val_metric:.4f} at epoch {best_epoch+1}")
        
        # 加载最佳模型
        if hasattr(self, 'best_model_state'):
            self.net_.load_state_dict(self.best_model_state)
        
        return self
    
    # 修改train_one_epoch方法，添加梯度裁剪
    def train_one_epoch(self, train_loader, epoch):
        """训练一个周期"""
        self.net_.train()
        torch.set_grad_enabled(True)
        
        all_losses = []
        all_outputs = []
        all_targets = []
        
        # 初始化ranking loss参数
        lambda_coeff = 0.005
        margin = 0.25
        margin_loss = torch.nn.MarginRankingLoss(reduction='sum', margin=margin)
        
        for x_batch, y_batch, mask, y_mask in train_loader:
            # 转移数据到设备
            x_batch = {k: x_batch[k].to(self.device) for k in x_batch}
            y_batch = {k: y_batch[k].to(self.device) for k in y_batch}
            mask = {k: mask[k].to(self.device) for k in mask}
            y_mask = {k: y_mask[k].to(self.device) for k in y_mask}
            
            # 前向传播
            outputs = self.net_(x_batch, mask)
            
            # 计算focal loss
            loss = 0
            for k in self.tgt_modalities:
                # 应用标签平滑
                if self.label_smoothing > 0:
                    targets = y_batch[k] * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
                else:
                    targets = y_batch[k]
                    
                loss_task = self.loss_fn[k](outputs[k], targets)
                msk_loss_task = loss_task * y_mask[k]
                msk_loss_mean = msk_loss_task.sum() / y_mask[k].sum()
                loss += msk_loss_mean
                
                # 收集输出和目标
                all_outputs.extend(torch.sigmoid(outputs[k]).detach().cpu().numpy())
                all_targets.extend(y_batch[k].detach().cpu().numpy())
            
            # 添加ranking loss (仅在有多个预测目标时)
            if len(self.tgt_modalities) > 1 and epoch >= 10:  # 前10个epoch只使用focal loss以稳定训练
                for i, k1 in enumerate(self.tgt_modalities):
                    for j, k2 in enumerate(self.tgt_modalities):
                        if j > i:  # 只处理唯一的对
                            # 找出两个任务都有标签的样本
                            pairs = (y_mask[k1] == 1) & (y_mask[k2] == 1)
                            total_elements = (torch.abs(y_batch[k1][pairs] - y_batch[k2][pairs])).sum()
                            
                            if total_elements != 0:
                                # 添加ranking loss
                                ranking_loss = lambda_coeff * (
                                    margin_loss(
                                        torch.sigmoid(outputs[k1])[pairs],
                                        torch.sigmoid(outputs[k2])[pairs],
                                        y_batch[k1][pairs] - y_batch[k2][pairs]
                                    )
                                ) / total_elements
                                loss += ranking_loss
            
            # 反向传播
            self.optimizer.zero_grad()
            loss.backward()
            
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(self.net_.parameters(), max_norm=1.0)
            
            self.optimizer.step()
            
            all_losses.append(loss.item())
        
        # 计算指标
        if len(all_targets) > 0:
            try:
                auc_score = roc_auc_score(all_targets, all_outputs)
            except:
                auc_score = 0.5
        else:
            auc_score = 0.5
            
        metrics = {
            'Loss': np.mean(all_losses),
            'AUC': auc_score
        }
        
        return metrics
    
    def validate_one_epoch(self, val_loader):
        """验证一个周期"""
        self.net_.eval()
        torch.set_grad_enabled(False)
        
        all_losses = []
        all_outputs = []
        all_targets = []
        
        for x_batch, y_batch, mask, y_mask in val_loader:
            # 转移数据到设备
            x_batch = {k: x_batch[k].to(self.device) for k in x_batch}
            y_batch = {k: y_batch[k].to(self.device) for k in y_batch}
            mask = {k: mask[k].to(self.device) for k in mask}
            y_mask = {k: y_mask[k].to(self.device) for k in y_mask}
            
            # 前向传播
            outputs = self.net_(x_batch, mask)
            
            # 计算损失
            for k in self.tgt_modalities:
                loss_task = self.loss_fn[k](outputs[k], y_batch[k])
                msk_loss_task = loss_task * y_mask[k]
                msk_loss_mean = msk_loss_task.sum() / y_mask[k].sum()
                
                # 收集损失
                all_losses.append(msk_loss_mean.item())
                
                # 收集输出和目标
                all_outputs.extend(torch.sigmoid(outputs[k]).detach().cpu().numpy())
                all_targets.extend(y_batch[k].detach().cpu().numpy())
        
        # 计算指标
        precision, recall, _ = precision_recall_curve(all_targets, all_outputs)
        metrics = {
            'Loss': np.mean(all_losses),
            'AUC': roc_auc_score(all_targets, all_outputs),
            'AUC (ROC)': roc_auc_score(all_targets, all_outputs),
            'AUC (PR)': auc(recall, precision)
        }
        
        return metrics
    
    def predict(self, x_test):
        """预测新样本"""
        self.net_.eval()
        torch.set_grad_enabled(False)
        
        # 准备测试数据
        # 使用列表推导，为每个样本创建一个空的目标变量字典
        target_key = next(iter(self.tgt_modalities.keys()))  # 获取目标变量的键名
        dummy_targets = [{target_key: 0} for _ in range(len(x_test))]
        
        test_dataset = IrisDataset(
            x_test, 
            dummy_targets,  # 使用虚拟标签
            self.src_modalities,
            dropout_rate=0  # 预测时不使用dropout
        )
        
        test_loader = DataLoader(
            test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=IrisDataset.collate_fn
        )
        
        logits = []
        probas = []
        
        for x_batch, _, mask, _ in test_loader:
            # 转移数据到设备
            x_batch = {k: x_batch[k].to(self.device) for k in x_batch}
            mask = {k: mask[k].to(self.device) for k in mask}
            
            # 前向传播
            outputs = self.net_(x_batch, mask)
            
            # 收集输出
            batch_logits = {k: outputs[k].detach().cpu().numpy() for k in outputs}
            batch_probas = {k: torch.sigmoid(outputs[k]).detach().cpu().numpy() for k in outputs}
            
            # 转换为列表
            for i in range(len(next(iter(batch_logits.values())))):
                logits.append({k: batch_logits[k][i] for k in batch_logits})
                probas.append({k: batch_probas[k][i] for k in batch_probas})
        
        # 预测结果
        preds = [{k: int(p[k] > 0.5) for k in p} for p in probas]
        
        return logits, probas, preds


In [6]:
# 加载数据集
data = pd.read_csv('site_num_processed5.csv',index_col=0)
data = data[(data['Region_Code'] != 1) & (data['Region_Code'] != 9)]
    # 定义分类特征和数值特征
protein_cols = data.columns.tolist()[9:] # 数值特征
pred_cols = ['any_protein_pred','depressed_protein_pred','dementia_protein_pred','anxiety_protein_pred','sleep_protein_pred','sud_protein_pred','sd_protein_pred',
                   'any_bio_pred','depressed_bio_pred','dementia_bio_pred','anxiety_bio_pred','sleep_bio_pred','sud_bio_pred','sd_bio_pred',
                   'any_phe_pred','depressed_phe_pred','dementia_phe_pred','anxiety_phe_pred','sleep_phe_pred','sud_phe_pred','sd_phe_pred',
                   # 'any_prs_pred','depressed_prs_pred','dementia_prs_pred','anxiety_prs_pred','sleep_prs_pred','sud_prs_pred','sd_prs_pred',
                   'text_pred']
X = data[protein_cols]
y = data['any'] # 标签
    
    # 数据分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y, shuffle=True)
    
    # 标准化
scaler = StandardScaler()
#X_train = scaler.fit_transform(X_train[pred_cols])
tmp = scaler.fit_transform(X_train[pred_cols])
df_prot_train_tissue = pd.DataFrame(tmp, index=X_train[pred_cols].index, columns=X_train[pred_cols].columns)
other_cols = [col for col in X_train.columns if col not in pred_cols]
X_train = pd.concat([X_train[other_cols], df_prot_train_tissue], axis=1)    
X_train = X_train.values
#X_test = scaler.transform(X_test[pred_cols])
tmp = scaler.transform(X_test[pred_cols])
df_prot_train_tissue = pd.DataFrame(tmp, index=X_test[pred_cols].index, columns=X_test[pred_cols].columns)
other_cols = [col for col in X_test.columns if col not in pred_cols]
X_test = pd.concat([X_test[other_cols], df_prot_train_tissue], axis=1) 
X_test = X_test.values
# 定义源模态和目标模态
src_modalities = {}
for i, col in enumerate(protein_cols):
        src_modalities[f'feat_{i}'] = {'type': 'numerical', 'shape': 1}

tgt_modalities = {
        'is_depressed': {'type': 'binary', 'shape': 1}
    }
    
    # 标签比例
label_fractions = {'is_depressed': float(sum(y_train)/len(y_train))}
print(f"标签比例：{label_fractions}")
print(f"特征数量：{len(src_modalities)}")
    
    # 初始化模型
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SimpleADRDModel(
    src_modalities=src_modalities,
    tgt_modalities=tgt_modalities,
    label_fractions=label_fractions,
    d_model=32,  # 减小模型容量
    nhead=4,     # 减少注意力头
    num_encoder_layers=2,  # 减少层数
    num_epochs=30,  # 增加epoch但配合早停
    batch_size=32,  # 增大批次大小
    lr=1e-3,  # 降低学习率
    weight_decay=0.01,  # 增加权重衰减
    criterion='AUC (ROC)',
    device=device,
    verbose=1,
    dropout_rate=0.1,  # 特征dropout
    early_stopping_patience=8,  # 早停耐心值
    early_stopping_delta=0.001,  # 最小改善阈值
    dropout_transformer=0.1,  # Transformer内部dropout
    label_smoothing=0.1,  # 标签平滑
    warmup_epochs=5,  # 热身epoch
    lr_min=1e-6  # 最小学习率
)
    
    # 准备数据格式
x_train_dict = []
for row in X_train:
        sample = {}
        for i in range(len(row)):
            sample[f'feat_{i}'] = row[i]
        x_train_dict.append(sample)
    
x_test_dict = []
for row in X_test:
        sample = {}
        for i in range(len(row)):
            sample[f'feat_{i}'] = row[i]
        x_test_dict.append(sample)
    
y_train_dict = [{'is_depressed': int(label)} for label in y_train]
y_test_dict = [{'is_depressed': int(label)} for label in y_test]
    
    # 训练模型
model.fit(x_train_dict, x_test_dict, y_train_dict, y_test_dict)
    
# 预测
logits, probas, preds = model.predict(x_test_dict)
    
    # 评估结果
y_pred = np.array([p['is_depressed'] for p in preds])
accuracy = accuracy_score(y_test, y_pred)
auc_score = roc_auc_score(y_test, [p['is_depressed'] for p in probas])
    
print(f"\n测试集结果:")
print(f"准确率: {accuracy:.4f}")
print(f"AUC: {auc_score:.4f}")
import joblib
joblib.dump(scaler, f'model/tran_standard_scaler_mix2.joblib')
joblib.dump(model,f'model/tran_model_mix2.pkl')

标签比例：{'is_depressed': 0.17289362691375246}
特征数量：265


c:\Users\12225\.conda\envs\py311\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 1/30, LR: 0.000200
Train - Loss: 0.0614, AUC: 0.5106
Val   - Loss: 0.0598, AUC: 0.6749
Validation loss decreased (inf --> 0.059754). Saving model...
Epoch 2/30, LR: 0.000400
Train - Loss: 0.0500, AUC: 0.7849
Val   - Loss: 0.0552, AUC: 0.7740
Validation loss decreased (0.059754 --> 0.055158). Saving model...
Epoch 3/30, LR: 0.000600
Train - Loss: 0.0392, AUC: 0.8873
Val   - Loss: 0.0638, AUC: 0.7787
EarlyStopping counter: 1 out of 8
Epoch 4/30, LR: 0.000800
Train - Loss: 0.0367, AUC: 0.9035
Val   - Loss: 0.0672, AUC: 0.7805
EarlyStopping counter: 2 out of 8
Epoch 5/30, LR: 0.001000
Train - Loss: 0.0350, AUC: 0.9136
Val   - Loss: 0.0590, AUC: 0.7841
EarlyStopping counter: 3 out of 8
Epoch 6/30, LR: 0.000996
Train - Loss: 0.0345, AUC: 0.9165
Val   - Loss: 0.0631, AUC: 0.7883
EarlyStopping counter: 4 out of 8
Epoch 7/30, LR: 0.000984
Train - Loss: 0.0332, AUC: 0.9225
Val   - Loss: 0.0758, AUC: 0.7868
EarlyStopping counter: 5 out of 8
Epoch 8/30, LR: 0.000965
Train - Loss: 0.0324, AUC

['model/tran_model_mix2.pkl']

In [ ]:
import joblib
scaler = joblib.load('model/tran_standard_scaler_mix2.joblib')
model = joblib.load('model/tran_model_mix2.pkl')
data = pd.read_csv('site_num_processed4.csv',index_col=0)
data_ex = data[(data['Region_Code'] == 1) | (data['Region_Code'] == 9)]
X = data_ex[protein_cols]
y = data_ex['any'] # 标签
tmp = scaler.transform(X[pred_cols])
df_prot_train_tissue = pd.DataFrame(tmp, index=X[pred_cols].index, columns=X[pred_cols].columns)
other_cols = [col for col in X.columns if col not in pred_cols]
X = pd.concat([X[other_cols], df_prot_train_tissue], axis=1)    
X = X.values
# 定义源模态和目标模态
src_modalities = {}
for i, col in enumerate(protein_cols):
    src_modalities[f'feat_{i}'] = {'type': 'numerical', 'shape': 1}

tgt_modalities = {
    'is_depressed': {'type': 'binary', 'shape': 1}
}
label_fractions = {'is_depressed': float(sum(y)/len(y))}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
x_dict = []
for row in X:
    sample = {}
    for i in range(len(row)):
        sample[f'feat_{i}'] = row[i]
    x_dict.append(sample)
y_dict = [{'is_depressed': int(label)} for label in y]
logits, probas_ex, preds = model.predict(x_dict)
    
# 评估结果
y_pred = np.array([p['is_depressed'] for p in preds])
accuracy = accuracy_score(y, y_pred)
auc_score = roc_auc_score(y, [p['is_depressed'] for p in probas_ex])
mu_y_pred_proba_ex = np.array([p['is_depressed'] for p in probas_ex])
auc_score 